# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
url = "hf://datasets/FlyRank/internship-warehouse"
sample_rel = f"{url}/fact_content_daily_performance_sample.parquet"
rel = f"{url}/fact_content_daily_performance/**/*.parquet"

con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The method I choose to use is the "yes/no" with observed label using logistic regression. This label attempts to a answer a question of: Did a declining page recover by the next time period? This fits the refresh/content opportunity scoring lane because it studies the refresh potential of a declining page. To start, logistic regression and random forest will be used. Features will be selected according to the data contract and multicollinearity will be researched. The output model's coefficients will provide explanations to important features. Boost methods will be considered to further enhance the model.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Training can only occur on client data with enough history. A cutoff date T will be selected and tuned to train the model using future (after T) recovery data and past (before T) decline data. Clients are to be split into train/test using 5-fold cross validation.

### Picking date split

To select a cutoff date T, we need to find eligible clients (clients with at least 2 months of data before and after) for each date option. The date of interest is the earliest date with the highest amount of eligible clients.

In [2]:
import pandas as pd

WINDOW_DAYS = 30       # comparison window size (last30/prev30)
LABEL_GAP_DAYS = 1     # days between T and the start of the recovery-check window
LABEL_WINDOW_DAYS = 30 # size of the recovery-check window

date_range = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}')
""").df()
earliest, latest = date_range["earliest"][0], date_range["latest"][0]

candidate_Ts = pd.date_range(earliest, latest, freq="MS")

scan_results = []
for T_candidate in candidate_Ts:
    T_str = T_candidate.strftime("%Y-%m-%d")
    n = con.sql(f"""
        WITH client_ranges AS (
            SELECT client_hash_id, MIN(report_date) AS min_date, MAX(report_date) AS max_date
            FROM read_parquet('{rel}')
            GROUP BY client_hash_id
        )
        SELECT COUNT(DISTINCT CASE WHEN min_date <= (DATE '{T_str}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                    AND max_date >= (DATE '{T_str}' + INTERVAL {2*LABEL_WINDOW_DAYS + LABEL_GAP_DAYS} DAY)
                               THEN client_hash_id END) AS n
        FROM client_ranges
    """).df()["n"][0]
    scan_results.append({"T": T_str, "eligible_clients": n})

scan_df = pd.DataFrame(scan_results)
print(scan_df)

T = scan_df.loc[scan_df['eligible_clients'].idxmax(), 'T']
print(f"\nBest T: {T} "
      f"with {scan_df['eligible_clients'].max()} eligible clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

             T  eligible_clients
0   2025-02-01                 0
1   2025-03-01                 0
2   2025-04-01                 2
3   2025-05-01                 3
4   2025-06-01                 4
5   2025-07-01                 4
6   2025-08-01                 4
7   2025-09-01                10
8   2025-10-01                16
9   2025-11-01                16
10  2025-12-01                24
11  2026-01-01                32
12  2026-02-01                39
13  2026-03-01                39
14  2026-04-01                38
15  2026-05-01                 0
16  2026-06-01                 0

Best T: 2026-02-01 with 39 eligible clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# --- feature set, as of T only, no label-derived fields ---
content_type_query = f"""
    SELECT content_hash_id, keyword_char_count, url_char_count, content_type,
        search_volume, competition, main_intent, category_count, model_used, char_count,
        DATEDIFF('day', content_updated_date, DATE '{T}') AS days_since_last_update,
        DATEDIFF('day', content_created_date, DATE '{T}') AS days_since_created
    FROM read_parquet('{url}/dim_content.parquet')
"""
content_dim = con.sql(content_type_query).df()

feature_query = f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{rel}')
    WHERE report_date <= DATE '{T}'
    GROUP BY client_hash_id, content_hash_id
"""
X_raw = con.sql(feature_query).df().merge(content_dim, on="content_hash_id", how="left")

# --- gate: is_declining at T ---
gate_query = f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 < -20
             THEN 1 ELSE 0 END AS is_declining_at_T
    FROM windowed
"""
gate_df = con.sql(gate_query).df()

# --- recovery label: strictly after T ---
future_query = f"""
    WITH future AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_T1,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_baseline
        FROM read_parquet('{rel}')
        WHERE report_date > DATE '{T}'
          AND report_date <= (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN impr_T1 > impr_baseline THEN 1 ELSE 0 END AS recovered_by_T1
    FROM future
"""
recovery_df = con.sql(future_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Data handling

#### Impute missing data

In [4]:
print("Before imputing:\n", X_raw.isna().sum())

Before imputing:
 client_hash_id                 0
content_hash_id                0
gsc_avg_position          115070
gsc_impressions                0
gsc_clicks                     0
keyword_char_count             0
url_char_count                 0
content_type                   0
search_volume              57071
competition                57071
main_intent                55417
category_count                 0
model_used                 84060
char_count                105346
days_since_last_update         0
days_since_created             0
dtype: int64


In [5]:
# Search volume and competition have the same missing count, combine into one missing flag
X_raw["has_keyword_data"] = X_raw["search_volume"].notna() & X_raw["competition"].notna()
X_raw["char_count_imputed"] = X_raw["char_count"].isna()
X_raw["ai_generated"] = X_raw["model_used"].notna()

X_raw["gsc_avg_position"] = X_raw["gsc_avg_position"].fillna(0) # 0 means no position data
X_raw["search_volume"] = X_raw["search_volume"].fillna(0)
X_raw["competition"] = X_raw["competition"].fillna(0)
X_raw["main_intent"] = X_raw["main_intent"].fillna("no_keyword")
X_raw["model_used"] = X_raw["model_used"].fillna("human")
X_raw["char_count"] = X_raw["char_count"].fillna(X_raw["char_count"].median())

In [79]:
print("After imputing:\n", X_raw.isna().sum())

After imputing:
 client_hash_id            0
content_hash_id           0
gsc_avg_position          0
gsc_impressions           0
gsc_clicks                0
keyword_char_count        0
url_char_count            0
content_type              0
search_volume             0
competition               0
main_intent               0
category_count            0
model_used                0
char_count                0
days_since_last_update    0
days_since_created        0
has_keyword_data          0
char_count_imputed        0
ai_generated              0
dtype: int64


### Model Building

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

declining_ids = gate_df.loc[gate_df["is_declining_at_T"] == 1, ["client_hash_id", "content_hash_id"]]

data = (X_raw.merge(declining_ids, on=["client_hash_id", "content_hash_id"], how="inner")
             .merge(recovery_df[["client_hash_id", "content_hash_id", "recovered_by_T1"]],
                     on=["client_hash_id", "content_hash_id"], how="inner"))

y = data.pop("recovered_by_T1")
groups = data["client_hash_id"]
categorical_text_cols = ["content_type", "main_intent", "model_used"]
X_full = pd.get_dummies(data.drop(columns=["client_hash_id", "content_hash_id"]), columns=categorical_text_cols)

param_grid = {"max_depth": [2, 3, 4, 6, 8, None]}
grid = GridSearchCV(
    RandomForestClassifier(random_state=33, class_weight="balanced"),
    param_grid, cv=GroupKFold(n_splits=5), scoring="roc_auc"
)
grid.fit(X_full, y, groups=groups)
print(f"Best depth: {grid.best_params_}, AUC: {grid.best_score_:.3f}")

Best depth: {'max_depth': 8}, AUC: 0.706


In [11]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

def precision_at_k(scores, labels, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_group_kfold(X, y, groups, label):
    n_clients = groups.nunique()
    n_folds = min(5, n_clients)

    gkf = GroupKFold(n_splits=n_folds)
    fold_results = []
    coef_records = []
    importance_records = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

        logit = LogisticRegression(random_state=33, class_weight="balanced", max_iter=2000).fit(X_train_scaled, y_train)
        rf = RandomForestClassifier(max_depth=8, random_state=33, class_weight="balanced").fit(X_train, y_train)

        coef_records.append(pd.Series(logit.coef_[0], index=X_train.columns, name=f"fold_{fold}"))
        importance_records.append(pd.Series(rf.feature_importances_, index=X_train.columns, name=f"fold_{fold}"))

        stayed_broken = 1 - y_test.values
        logit_scores = 1 - logit.predict_proba(X_test_scaled)[:, 1]
        rf_scores = 1 - rf.predict_proba(X_test)[:, 1]

        fold_results.append({
            "fold": fold, "test_clients": groups.iloc[test_idx].nunique(), "test_rows": len(y_test),
            "recovered_rate": y_test.mean(),
            "logit_auc": roc_auc_score(y_test, logit.predict_proba(X_test_scaled)[:, 1]),
            "rf_auc": roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]),
            "logit_p@20": precision_at_k(logit_scores, stayed_broken, 20),
            "rf_p@20": precision_at_k(rf_scores, stayed_broken, 20),
            "logit_p@50": precision_at_k(logit_scores, stayed_broken, 50),
            "rf_p@50": precision_at_k(rf_scores, stayed_broken, 50),
            "base_rate": stayed_broken.mean(),
        })

    fold_df = pd.DataFrame(fold_results)
    fold_df["logit_p@20_lift"] = fold_df["logit_p@20"] - fold_df["base_rate"]
    fold_df["rf_p@20_lift"] = fold_df["rf_p@20"] - fold_df["base_rate"]
    fold_df["logit_p@50_lift"] = fold_df["logit_p@50"] - fold_df["base_rate"]
    fold_df["rf_p@50_lift"] = fold_df["rf_p@50"] - fold_df["base_rate"]

    metric_cols = ["logit_auc", "rf_auc", "logit_p@20", "logit_p@50", "rf_p@20", "rf_p@50", "logit_p@20_lift", "rf_p@20_lift", "logit_p@50_lift", "rf_p@50_lift"]
    summary = fold_df[metric_cols].agg(["mean", "std"]).T
    summary.columns = [f"{label}_mean", f"{label}_std"]

    coef_df = pd.concat(coef_records, axis=1)
    coef_summary = pd.DataFrame({"mean": coef_df.mean(axis=1), "std": coef_df.std(axis=1)}).sort_values("mean", key=abs, ascending=False)

    importance_df = pd.concat(importance_records, axis=1)
    importance_summary = pd.DataFrame({"mean": importance_df.mean(axis=1), "std": importance_df.std(axis=1)}).sort_values("mean", ascending=False)

    return {
        "n_clients": n_clients, "n_folds": n_folds, "n_features": X.shape[1],
        "fold_df": fold_df, "summary": summary,
        "coef_summary": coef_summary, "importance_summary": importance_summary,
    }

# --- run 1: all features ---
results_full = run_group_kfold(X_full, y, groups, label="full")

print(f"Full feature set: {results_full['n_features']} features")
print(results_full["summary"])
print(results_full["coef_summary"])

Full feature set: 27 features
                 full_mean  full_std
logit_auc         0.684086  0.126087
rf_auc            0.706029  0.108479
logit_p@20        0.790000  0.151658
logit_p@50        0.856000  0.121161
rf_p@20           0.900000  0.127475
rf_p@50           0.904000  0.121984
logit_p@20_lift   0.192420  0.176695
rf_p@20_lift      0.302420  0.197470
logit_p@50_lift   0.258420  0.165301
rf_p@50_lift      0.306420  0.188606
                                       mean       std
days_since_last_update            -0.534783  0.129807
model_used_gpt-5-mini             -0.450021  0.144093
model_used_gemini-3-flash-preview  0.364285  0.086534
char_count_imputed                 0.348852  0.102501
gsc_impressions                    0.185935  0.173417
char_count                        -0.175839  0.077625
has_keyword_data                   0.149887  0.072783
gsc_clicks                        -0.144320  0.127633
days_since_created                 0.129081  0.030244
gsc_avg_position       

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Raw precision looks strong on its own: logit p@20 0.790 ± 0.152, rf p@20 0.910 ± 0.124,
rf p@50 0.884 ± 0.146. Read without context, over 90% precision@20 looks like a very
strong model. However,

1. **Lift over each fold's own base rate tells a different story than the raw numbers.**
   Base rate varies a lot fold to fold, so raw precision@K is partly just reflecting how
   common "stayed broken" already was in that fold, not the model's own contribution.
   The best result (rf_p@20_lift) is, on average, about 31 percentage points better than
   base-rate guessing — a real improvement, but std (0.175) is still more than half the
   mean, so some folds show a much smaller gain than others.
2. **logit_p@20_lift has std (0.177) almost as large as its mean (0.192)** Close to half
   the folds could be showing close-to-no improvement over guessing, even though the
   average looks solid.

days_since_last_update (-0.535, std 0.130) is
the clear standout — small std relative to its mean.
model_used_gpt-5-mini (-0.450) and model_used_gemini-3-flash-preview (0.364) look like
the next-strongest effects, with char_count_imputed (0.349) close behind. content_type
and main_intent are both close to 0.

### Multicollinearity Check

In [7]:
!pip install phik --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 9.8 MB/s eta 0:00:00


In [12]:
import phik

cols_to_check = [c for c in X_raw.columns if c not in ["client_hash_id", "content_hash_id"]]
phik_matrix = X_raw[cols_to_check].phik_matrix()

phik_flat = phik_matrix.copy()
np.fill_diagonal(phik_flat.values, np.nan)  # drop self-correlations (always 1.0)

phik_pairs = phik_flat.unstack().dropna().sort_values(ascending=False)
phik_pairs = phik_pairs[phik_pairs.index.get_level_values(0) < phik_pairs.index.get_level_values(1)]  # drop mirror duplicates

threshold = 0.5
strong_pairs = phik_pairs[phik_pairs > threshold]

# --- fix: flatten the MultiIndex into plain columns before printing ---
strong_pairs = strong_pairs.reset_index()
strong_pairs.columns = ["col1", "col2", "phik"]
strong_pairs = strong_pairs.sort_values("phik", ascending=False)

print(strong_pairs.to_string(index=False))

interval columns not set, guessing: ['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'keyword_char_count', 'url_char_count', 'search_volume', 'competition', 'category_count', 'char_count', 'days_since_last_update', 'days_since_created']
                  col1                   col2     phik
          ai_generated             model_used 1.000000
      has_keyword_data     keyword_char_count 0.999283
    char_count_imputed             model_used 0.983298
          ai_generated     char_count_imputed 0.967512
      has_keyword_data            main_intent 0.866435
    keyword_char_count            main_intent 0.837076
    char_count_imputed     days_since_created 0.774066
          content_type     keyword_char_count 0.752969
      has_keyword_data             model_used 0.689322
            gsc_clicks        gsc_impressions 0.683798
    days_since_created       has_keyword_data 0.650822
          content_type            main_intent 0.648900
    days_since_created            main_inte

#### Key Findings
1. `ai_generated` / `model_used` (1.00) and `has_keyword_data` / `keyword_char_count`
  (0.999) are expected: both flags are directly derived from their source
  column's missingness. Not redundant.

2. `char_count_imputed` / `model_used` (0.98) and `ai_generated` / `char_count_imputed`
  (0.97): char_count's missingness is concentrated in specific model_used
  categories. Not random.

3. `main_intent` / `keyword_char_count` (0.84) and
  `content_type` / `keyword_char_count` (0.75): both driven by intent
  and format largely determining keyword length.

4. `gsc_impressions` / `gsc_clicks` (0.68): Create a
  derived `ctr = gsc_clicks / gsc_impressions` to represent how well a page
  converts.

5. `days_since_created` associate broadly (0.40-0.77)
  with nearly every other column from (2), potentially tracking which content-production era a page came from.

6. `days_since_last_update` shows moderate correlation
  (0.36-0.51) with model_used, has_keyword_data, main_intent, url_char_count,
  char_count_imputed.

After checking correlations, a cluster of features: model_used,
char_count, char_count_imputed, and ai_generated may indicate one signal counted four times. Let's try PCA to reduce correlation and Elastic net for regularization.

In [13]:
from sklearn.decomposition import PCA

# Revision 1 (PCA variant): compress the confound cluster instead of dropping it,
# keep every clean feature untouched
confound_cols = [c for c in X_full.columns
                  if any(c.startswith(p) for p in
                         ["model_used", "char_count", "ai_generated"])]
clean_cols = [c for c in X_full.columns if c not in confound_cols]

scaler_confound = StandardScaler()
X_confound_scaled = scaler_confound.fit_transform(X_full[confound_cols])

pca = PCA(n_components=0.95)
X_confound_pca = pca.fit_transform(X_confound_scaled)
pca_cols = [f"confound_pc{i+1}" for i in range(X_confound_pca.shape[1])]
X_pca_df = pd.DataFrame(X_confound_pca, columns=pca_cols, index=X_full.index)

X_r1 = pd.concat([X_full[clean_cols].reset_index(drop=True),
                       X_pca_df.reset_index(drop=True)], axis=1)

print(f"Revision 1 (PCA): {X_r1.shape[1]} features "
      f"({len(clean_cols)} clean + {len(pca_cols)} PCA, dropped {len(confound_cols)} raw confound columns)")

results_r1 = run_group_kfold(X_r1, y, groups, label="r1_pca")

comparison = pd.concat([
    results_full["summary"],
    results_r1["summary"]
], axis=1)

print(comparison.round(3))

Revision 1 (PCA): 24 features (18 clean + 6 PCA, dropped 9 raw confound columns)
                 full_mean  full_std  r1_pca_mean  r1_pca_std
logit_auc            0.684     0.126        0.681       0.129
rf_auc               0.706     0.108        0.705       0.104
logit_p@20           0.790     0.152        0.800       0.150
logit_p@50           0.856     0.121        0.856       0.124
rf_p@20              0.900     0.127        0.890       0.114
rf_p@50              0.904     0.122        0.880       0.150
logit_p@20_lift      0.192     0.177        0.202       0.192
rf_p@20_lift         0.302     0.197        0.292       0.158
logit_p@50_lift      0.258     0.165        0.258       0.162
rf_p@50_lift         0.306     0.189        0.282       0.215


In [14]:
scaler_final = StandardScaler()
X_r1_scaled = pd.DataFrame(scaler_final.fit_transform(X_r1), columns=X_r1.columns)
logit_r1_pca = LogisticRegression(class_weight="balanced", max_iter=2000).fit(X_r1_scaled, y)
coefs_r1_pca = pd.Series(logit_r1_pca.coef_[0], index=X_r1.columns).sort_values(key=abs, ascending=False)
print(coefs_r1_pca)

days_since_last_update         -0.610600
confound_pc3                   -0.472006
confound_pc1                    0.448360
days_since_created              0.246289
gsc_impressions                 0.156810
confound_pc6                   -0.139242
gsc_clicks                     -0.132996
confound_pc2                    0.126602
confound_pc4                    0.116644
has_keyword_data                0.114984
url_char_count                 -0.112651
gsc_avg_position                0.078417
main_intent_transactional      -0.068682
content_type_feedly article    -0.057761
content_type_keyword article    0.057761
main_intent_no_keyword          0.044057
competition                    -0.039445
main_intent_commercial         -0.037506
main_intent_informational       0.034014
confound_pc5                   -0.026813
category_count                  0.023456
search_volume                   0.023171
main_intent_navigational        0.017160
keyword_char_count              0.003470
dtype: float64


In [15]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler

# Revision 2 (Elastic Net variant): compress the confound cluster instead of dropping it,
# keep every clean feature untouched

scaler_elastic = StandardScaler()
X_full_scaled_elastic = pd.DataFrame(scaler_elastic.fit_transform(X_full), columns=X_full.columns)

cv_splits_full = list(GroupKFold(n_splits=5).split(X_full_scaled_elastic, y, groups=groups))

elastic = LogisticRegressionCV(
    penalty="elasticnet",
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    solver="saga",
    cv=cv_splits_full,
    scoring="roc_auc",
    class_weight="balanced",
    max_iter=3000,
    random_state=33,
)
elastic.fit(X_full_scaled_elastic, y)

elastic_coefs = pd.Series(elastic.coef_[0], index=X_full.columns).sort_values(key=abs, ascending=False)
print(elastic_coefs)
print(f"\nBest l1_ratio: {elastic.l1_ratio_[0]}, C: {elastic.C_[0]:.4f}")

days_since_last_update              -0.514889
model_used_gpt-5-mini               -0.439723
char_count_imputed                   0.344630
model_used_gemini-3-flash-preview    0.272870
days_since_created                   0.123274
model_used_gpt-4o-mini              -0.111184
char_count                          -0.094356
main_intent_informational            0.085726
url_char_count                      -0.081397
has_keyword_data                     0.081341
gsc_avg_position                     0.068967
ai_generated                        -0.058555
model_used_human                     0.058555
gsc_impressions                      0.048469
gsc_clicks                          -0.027513
main_intent_transactional           -0.004395
search_volume                        0.000445
keyword_char_count                   0.000000
competition                          0.000000
category_count                       0.000000
main_intent_commercial               0.000000
content_type_keyword article      

In [22]:
elastic_selected = elastic_coefs[abs(elastic_coefs) > 0.001].index.tolist()
X_elastic = X_full[elastic_selected]
results_r2 = run_group_kfold(X_elastic, y, groups, label="r2_en")

print(f"\nR2 (Elastic Net): {X_elastic.shape[1]} features")

comparison = pd.concat([
    results_full["summary"],
    results_r1["summary"],
    results_r2["summary"]
], axis=1)

print(comparison.round(3))

print(results_r2["coef_summary"])


R2 (Elastic Net): 16 features
                 full_mean  full_std  r1_pca_mean  r1_pca_std  r2_en_mean  \
logit_auc            0.684     0.126        0.681       0.129       0.692   
rf_auc               0.706     0.108        0.705       0.104       0.708   
logit_p@20           0.790     0.152        0.800       0.150       0.790   
logit_p@50           0.856     0.121        0.856       0.124       0.848   
rf_p@20              0.900     0.127        0.890       0.114       0.930   
rf_p@50              0.904     0.122        0.880       0.150       0.912   
logit_p@20_lift      0.192     0.177        0.202       0.192       0.192   
rf_p@20_lift         0.302     0.197        0.292       0.158       0.332   
logit_p@50_lift      0.258     0.165        0.258       0.162       0.250   
rf_p@50_lift         0.306     0.189        0.282       0.215       0.314   

                 r2_en_std  
logit_auc            0.123  
rf_auc               0.103  
logit_p@20           0.152  
logit

Compared to PCA, elastic net uses a combination of lasso and ridge regression to regularize components of the cluster. The cross-validated search optimizing for AUC prefers a model that relies on model_used/char_count signals over content_type which is fully removed in elastic net. Elastic net is very slightly ahead in AUC though both land in the 0.68-0.70 range. Elastic net is preferred for simpler interpretation of its original coefficients and direct handling of redundancy such as keyword_char_count.

In [19]:
# clean version, given gsc_impressions is already in elastic_selected
clean_cols_r3 = [c for c in elastic_selected if c != "gsc_clicks"]
X_r3_input = X_full[clean_cols_r3].copy()

X_r3_input["ctr"] = data["gsc_clicks"] / data["gsc_impressions"].replace(0, np.nan)
X_r3_input["ctr"] = X_r3_input["ctr"].fillna(0)

scaler_r3 = StandardScaler()
X_r3_input_scaled = pd.DataFrame(scaler_r3.fit_transform(X_r3_input), columns=X_r3_input.columns)

cv_splits_r3 = list(GroupKFold(n_splits=5).split(X_r3_input_scaled, y, groups=groups))

elastic_r3 = LogisticRegressionCV(
    penalty="elasticnet",
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    solver="saga",
    cv=cv_splits_r3,
    scoring="roc_auc",
    class_weight="balanced",
    max_iter=3000,
    random_state=33,
)

elastic_r3.fit(X_r3_input_scaled, y)


LogisticRegressionCV(class_weight='balanced',
                     cv=[(array([    0,     1,     2, ..., 34399, 34400, 34401]),
                          array([ 2951,  2952,  2953, ..., 33868, 33869, 33870])),
                         (array([    0,     1,     2, ..., 34399, 34400, 34401]),
                          array([ 2243,  2244,  2245, ..., 34160, 34161, 34162])),
                         (array([    0,     1,     2, ..., 34399, 34400, 34401]),
                          array([  524,   525,   526, ..., 33975, 33976, 33977])),
                         (array([  524,   525,   526, ..., 34202, 34203, 34204]),
                          array([    0,     1,     2, ..., 34399, 34400, 34401])),
                         (array([    0,     1,     2, ..., 34399, 34400, 34401]),
                          array([  650,   651,   652, ..., 34202, 34203, 34204]))],
                     l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9], max_iter=3000,
                     penalty='elasticnet', random_state=33, scoring='roc_auc',
                     solver='saga')

In [20]:
elastic_r3_coefs = pd.Series(elastic_r3.coef_[0], index=X_r3_input.columns).sort_values(key=abs, ascending=False)
r3_selected = elastic_r3_coefs[abs(elastic_r3_coefs) > 1e-6].index.tolist()
X_r3 = X_r3_input[r3_selected]

print(f"\nR3 (Elastic Net + ctr): {X_r3.shape[1]} features")
results_r3 = run_group_kfold(X_r3, y, groups, label="r3_ctr")

comparison = pd.concat([
    results_full["summary"],
    results_r1["summary"],
    results_r2["summary"],
    results_r3["summary"]
], axis=1)

print(comparison.round(3))

print(results_r3["coef_summary"])


R3 (Elastic Net + ctr): 16 features
                 full_mean  full_std  r1_pca_mean  r1_pca_std  r2_en_mean  \
logit_auc            0.684     0.126        0.681       0.129       0.692   
rf_auc               0.706     0.108        0.705       0.104       0.708   
logit_p@20           0.790     0.152        0.800       0.150       0.790   
logit_p@50           0.856     0.121        0.856       0.124       0.848   
rf_p@20              0.900     0.127        0.890       0.114       0.930   
rf_p@50              0.904     0.122        0.880       0.150       0.912   
logit_p@20_lift      0.192     0.177        0.202       0.192       0.192   
rf_p@20_lift         0.302     0.197        0.292       0.158       0.332   
logit_p@50_lift      0.258     0.165        0.258       0.162       0.250   
rf_p@50_lift         0.306     0.189        0.282       0.215       0.314   

                 r2_en_std  r3_ctr_mean  r3_ctr_std  
logit_auc            0.123        0.695       0.119  
rf_auc 

After addressing the listed multicollinearity problems, we achieve an AUC of 0.695 and 0.702 for logistic regression and random forest respectively. The precision values for both models @20 and 50 are around the 0.830-0.910 range. Random forest outperforms logistic regression on both AUC and precision.

In [24]:
from sklearn.ensemble import GradientBoostingRegressor

def run_boosted_kfold(X, y, groups, base_model_fn, correction_model_fn, label, n_folds=5):
    gkf = GroupKFold(n_splits=n_folds)
    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        base_model = base_model_fn()
        base_model.fit(X_train, y_train)
        base_train_probs = base_model.predict_proba(X_train)[:, 1]
        base_test_probs = base_model.predict_proba(X_test)[:, 1]

        residuals_train = y_train.values - base_train_probs
        correction = correction_model_fn()
        correction.fit(X_train, residuals_train)

        corrected_test_probs = np.clip(base_test_probs + correction.predict(X_test), 0, 1)

        fold_results.append({
            "fold": fold,
            "base_auc": roc_auc_score(y_test, base_test_probs),
            "boosted_auc": roc_auc_score(y_test, corrected_test_probs),
        })

    fold_df = pd.DataFrame(fold_results)
    print(f"\n{label}")
    print(f"  base:    AUC mean={fold_df['base_auc'].mean():.3f}, std={fold_df['base_auc'].std():.3f}")
    print(f"  boosted: AUC mean={fold_df['boosted_auc'].mean():.3f}, std={fold_df['boosted_auc'].std():.3f}")
    return fold_df

In [25]:
run_boosted_kfold(
    X_r3, y, groups,
    base_model_fn=lambda: RandomForestClassifier(max_depth=8, random_state=33, class_weight="balanced"),
    correction_model_fn=lambda: GradientBoostingRegressor(max_depth=3, n_estimators=20, learning_rate=0.1, random_state=33),
    label="RF + gradient-boosted correction"
)
run_boosted_kfold(
    X_r3, y, groups,
    base_model_fn=lambda: LogisticRegression(class_weight="balanced", max_iter=3000),
    correction_model_fn=lambda: GradientBoostingRegressor(max_depth=3, n_estimators=20, learning_rate=0.1, random_state=33),
    label="Logistic Regression + tree correction (same as before)"
)


RF + gradient-boosted correction
  base:    AUC mean=0.707, std=0.101
  boosted: AUC mean=0.706, std=0.097


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


Logistic Regression + tree correction (same as before)
  base:    AUC mean=0.687, std=0.125
  boosted: AUC mean=0.702, std=0.113


,fold,base_auc,boosted_auc
0,0,0.545110,0.583682
1,1,0.684887,0.700969
2,2,0.695413,0.698849
3,3,0.625526,0.643234
4,4,0.882192,0.885705


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.